<a href="https://colab.research.google.com/github/anjelhafidi2003-code/Phonetics-PhonologyFev2026/blob/main/WhisperSubtokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**install + build whisper.cpp (https://github.com/ggml-org/whisper.cpp**)

In [4]:
import os, subprocess, textwrap, sys

def run(cmd):
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

# system deps
run(["apt-get","-qq","update"])
run(["apt-get","-qq","install","-y","ffmpeg","cmake","build-essential","git"])

# clone + build
os.chdir("/content")
if not os.path.exists("whisper.cpp"):
    run(["git","clone","https://github.com/ggml-org/whisper.cpp"])
os.chdir("/content/whisper.cpp")

run(["cmake","-B","build"])
run(["cmake","--build","build","-j"])


apt-get -qq update
apt-get -qq install -y ffmpeg cmake build-essential git
cmake -B build
cmake --build build -j


 **download a multilingual model (could be base, small, medium, large)
!Attention: large model may crash Google Colab**

In [5]:
import os, subprocess

def run(cmd):
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


os.chdir("/content/whisper.cpp")
run(["bash","./models/download-ggml-model.sh","small"])  # You can change the model


bash ./models/download-ggml-model.sh small


**ensure WAV format (16k mono)**

In [6]:
import subprocess

WAV_IN  = "/content/arabic.wav"         #change the file path
WAV_16K = "/content/audio_16k.wav"

subprocess.run([
    "ffmpeg","-y","-i",WAV_IN,
    "-ar","16000","-ac","1","-c:a","pcm_s16le",
    WAV_16K
], check=True)

CompletedProcess(args=['ffmpeg', '-y', '-i', '/content/arabic.wav', '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', '/content/audio_16k.wav'], returncode=0)

**Transcribe + get tokens + (experimental) token timestamps + probs**

In [7]:
import os, subprocess, json
import re

os.chdir("/content/whisper.cpp")

MODEL = "models/ggml-small.bin"     #Be sure the model matches the one you selected in the previous cell.
AUDIO = "/content/audio_16k.wav"
OUT   = "/content/out"

subprocess.run(
    [
        "./build/bin/whisper-cli",
        "-m", MODEL,
        "-f", AUDIO,
        "-l", "auto",        # or "fr", "ar", ...
        "-ojf",
        "-dtw", "base",
        "-of", OUT
    ], check=True)

# Attempt to read the file with utf-8, replacing problematic characters
with open("/content/out.json", "r", encoding="utf-8", errors="replace") as f:
    json_str = f.read()

# Remove invalid control characters that might break JSON parsing.
# This regex matches any control character (0x00-0x1F) except tab (0x09), newline (0x0A), and carriage return (0x0D).
# These specific control characters are generally allowed in JSON strings if escaped.
# Other control characters are not allowed unescaped.
control_chars = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F]')
cleaned_json_str = control_chars.sub(lambda x: f'\\u{ord(x.group()):04x}', json_str)

# Now try parsing the cleaned string
j = json.loads(cleaned_json_str)

rows = []
for seg in j.get("transcription", []):
    for tok in seg.get("tokens", []):
        offs = tok.get("offsets", {})  # ms, when available
        rows.append({
            "token": tok.get("text"),
            "start_ms": offs.get("from"),
            "end_ms": offs.get("to"),
            "p": tok.get("p"),
            "id": tok.get("id"),
        })

__import__("pandas").DataFrame(rows).to_csv("/content/tokens.csv", index=False)

# print first tokens
for r in rows[:40]:
    print(r)

# print ALL tokens
print(f"Total tokens: {len(rows)}\n")
for i, r in enumerate(rows, 1):
    print(f"{i}. {r}")

{'token': '[_BEG_]', 'start_ms': 0, 'end_ms': 0, 'p': 0.575907, 'id': 50364}
{'token': ' است', 'start_ms': 140, 'end_ms': 450, 'p': 0.373852, 'id': 44713}
{'token': 'ح', 'start_ms': 590, 'end_ms': 590, 'p': 0.974548, 'id': 5016}
{'token': 'ب', 'start_ms': 710, 'end_ms': 730, 'p': 0.895557, 'id': 3555}
{'token': ' ر', 'start_ms': 730, 'end_ms': 840, 'p': 0.287072, 'id': 12602}
{'token': 'ج', 'start_ms': 920, 'end_ms': 1020, 'p': 0.985648, 'id': 7435}
{'token': 'ل', 'start_ms': 1020, 'end_ms': 1160, 'p': 0.985269, 'id': 1211}
{'token': ' ز', 'start_ms': 1160, 'end_ms': 1310, 'p': 0.892443, 'id': 30767}
{'token': 'وج', 'start_ms': 1310, 'end_ms': 1600, 'p': 0.998047, 'id': 29245}
{'token': 'ته', 'start_ms': 1600, 'end_ms': 1890, 'p': 0.930509, 'id': 47395}
{'token': ' لم', 'start_ms': 1890, 'end_ms': 2190, 'p': 0.781867, 'id': 32767}
{'token': 'ح', 'start_ms': 2190, 'end_ms': 2330, 'p': 0.989257, 'id': 5016}
{'token': 'ل', 'start_ms': 2330, 'end_ms': 2470, 'p': 0.701036, 'id': 1211}
{'tok

You can also manually download the `tokens.csv` file. Click on the folder icon on the left sidebar, navigate to `/content/tokens.csv`, right-click on the file, and select 'Download'.